# 03 · Zufallswege in Netzwerken: London und PageRank

Die Aufgaben sind in einzelne Stationen eingeteilt. Lies jeweils zuerst die Aufgabenstellung und mach dir eigene Gedanken und probiere dann das jeweilige Widget aus. Speichere deine Codeänderungen mit **Strg + S**.

---

Wie bisher gilt: **Spalte = Start, Zeile = Ziel**. Verteilungen sind Spaltenvektoren; ein Schritt bedeutet $p^{(k+1)}=Mp^{(k)}$.

**Schwierigkeitsstufen**

| Stufe | Orientierung |
|---|---|
| Leicht | Ab Klasse 9: Wahrscheinlichkeiten, Baumdiagramme, Bruchrechnen. Oft genügen Ausprobieren und genaues Lesen. |
| Schwerer | Ab Klasse 10/11: Matrizen, Gleichungssysteme, mehrstufige Überlegungen und Argumentieren mit Begriffen aus dem Skript. |
| Schwer | Eher Klasse 12/13: Knobelaufgaben, anspruchsvollere Begründungen, Beweisideen und ungewohnte Modellierung. |

Die Klassenstufen sind eine Orientierung, keine Zugangsvoraussetzung. Du kannst schwierigere Teilaufgaben auch überspringen, die Reihenfolge ist nicht obligatorisch.

<br>

## Vorbereitung der Widgets
#### Klicke zuerst oben im Menü auf **Run → Run All Cells**. Den Code in der hier folgenden Zelle brauchst du nicht zu bearbeiten.

In [ ]:
# Pakete und Widgets vorbereiten.
import sys

try:
    import ipywidgets as W
except ModuleNotFoundError:
    if sys.platform == "emscripten":
        import piplite

        await piplite.install("ipywidgets>=8,<9")
        import ipywidgets as W
    else:
        raise RuntimeError("Bitte ipywidgets in dieser Python-Umgebung installieren lassen.")


import io
import html
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from IPython.display import display
import base64

PANELS = {}


MINILINKS = {1: [2, 3], 2: [3], 3: [1], 4: []}


SKRIPTLINKS = {
    1: [2],
    2: [3],
    3: [2],
    4: [1, 2],
    5: [2, 4, 6],
    6: [2, 5],
    7: [2, 5],
    8: [2, 5],
    9: [5],
    10: [5],
}


# PageRank und gemeinsame Widget-Hilfen.
def entwicklung(M, p, n):
    result = [np.array(p, dtype=float)]
    for _ in range(n):
        result.append(M @ result[-1])
    return np.array(result)


def stationaer_eindeutig(M):
    """Keine willkürliche Auswahl bei mehreren stationären Verteilungen."""
    n = len(M)
    A = M - np.eye(n)
    if np.linalg.matrix_rank(A, tol=1e-10) != n - 1:
        return None
    A = A.copy()
    A[-1, :] = 1
    b = np.zeros(n)
    b[-1] = 1
    q = np.linalg.solve(A, b)
    if q.min() < -1e-10 or np.max(np.abs(M @ q - q)) > 1e-9:
        raise ValueError("Numerische Prüfung fehlgeschlagen; bitte ein anderes Modell wählen.")
    q = np.maximum(q, 0)
    return q / q.sum()


def linkmatrix(links):
    n = len(links)
    H = np.zeros((n, n))
    for j in range(1, n + 1):
        ziele = links[j]
        if ziele:
            H[np.array(ziele, dtype=int) - 1, j - 1] = 1 / len(ziele)
        else:
            H[:, j - 1] = 1 / n
    return H


def googlematrix(links, alpha):
    n = len(links)
    return alpha * linkmatrix(links) + (1 - alpha) * np.ones((n, n)) / n


def pfad_ziehen(M, start, n, seed):
    rng = np.random.default_rng(seed)
    grenzen = np.cumsum(M, axis=0)
    grenzen[-1, :] = 1
    uniforms = rng.random(n)
    x = np.empty(n + 1, dtype=int)
    x[0] = start
    for t, u in enumerate(uniforms, start=1):
        x[t] = np.searchsorted(grenzen[:, x[t - 1]], u, side="right")
    return x


def regler(value, maximum=100, minimum=0):
    return W.IntSlider(
        value=value,
        min=minimum,
        max=maximum,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )


def feld(text, w):
    return W.VBox([W.HTML(html.escape(text)), w])


def bild():
    return W.Image(format="png", layout=W.Layout(width="700px", max_width="100%"))


def zeichnen(fig, w):
    with io.BytesIO() as f:
        fig.savefig(f, format="png", dpi=105)
        w.value = f.getvalue()


def tabelle(head, rows):
    h = "".join(f'<th style="padding:4px 10px">{html.escape(str(x))}</th>' for x in head)
    body = "".join(
        "<tr>"
        + "".join(
            f'<td style="padding:4px 10px;text-align:center">{html.escape(str(x))}</td>'
            for x in row
        )
        + "</tr>"
        for row in rows
    )
    return '<div style="overflow-x:auto"><table><tr>' + h + "</tr>" + body + "</table></div>"


def matrixtext(M):
    return tabelle(
        ["Ziel ↓ / Start →"] + list(range(1, len(M) + 1)),
        [[i + 1] + [f"{v:.4f}" for v in row] for i, row in enumerate(M)],
    )


def netzbild(ax, links):
    ax.clear()
    size = len(links)
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, size, endpoint=False)
    pos = {j: (np.cos(a), np.sin(a)) for j, a in enumerate(angles, 1)}
    for j, targets in links.items():
        for i in targets:
            x, y = pos[j]
            if j == i:
                arrow = FancyArrowPatch(
                    (x - 0.12, y + 0.16),
                    (x + 0.12, y + 0.16),
                    connectionstyle="arc3,rad=-1.8",
                    arrowstyle="-|>",
                    mutation_scale=12,
                    color="#63758a",
                )
            else:
                arrow = FancyArrowPatch(
                    pos[j],
                    pos[i],
                    connectionstyle="arc3,rad=.14",
                    arrowstyle="-|>",
                    shrinkA=13,
                    shrinkB=13,
                    mutation_scale=12,
                    color="#63758a",
                )
            ax.add_patch(arrow)
    for j, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), 0.16, facecolor="#e5f0f8", edgecolor="#2875aa", zorder=3))
        ax.text(x, y, str(j), ha="center", va="center", zorder=4)
    ax.set(xlim=(-1.55, 1.55), ylim=(-1.4, 1.65), aspect="equal", title="Eingetragene Links")
    ax.axis("off")


def ranginfo(q):
    if q is None:
        return "Mehrere stationäre Verteilungen: keine eindeutige Rangliste."
    top = np.flatnonzero(np.isclose(q, q.max(), atol=1e-10, rtol=0)) + 1
    return "Größtes stationäres Gewicht: Seite(n) " + ", ".join(map(str, top)) + "."


# PageRank-Stationen.
def pagerank_labor(editierbar):
    size = 4 if editierbar else 10
    alpha = regler(85)
    steps = regler(10, 30, 0)
    steps.layout.width = "360px"
    start = W.Dropdown(options=list(range(1, size + 1)), value=1, description="Startseite:")
    pic = bild()
    info, table = W.HTML(), W.HTML()
    extra = W.Accordion(children=[table], selected_index=None)
    extra.set_title(0, "Matrix und Gewichte (aufklappen)")
    if editierbar:
        fig, (graphax, ax) = plt.subplots(
            1, 2, figsize=(6.7, 3.25), gridspec_kw={"width_ratios": [1, 1.35]}
        )
    else:
        fig, (graphax, ax) = plt.subplots(
            1, 2, figsize=(10, 4), gridspec_kw={"width_ratios": [1, 1.3]}
        )
        pic.layout.width = "1050px"
    x = np.arange(1, size + 1)
    barsq = ax.bar(x - 0.18, np.zeros(size), width=0.36, color="#2875aa", label="Stationäres q")
    barsp = ax.bar(
        x + 0.18,
        np.zeros(size),
        width=0.36,
        color="#ad6500",
        label="Verteilung nach k Schritten",
    )
    ax.set(
        xticks=x,
        xlabel="Seite",
        ylabel="Wahrscheinlichkeit",
        ylim=(0, 1.3),
        yticks=np.linspace(0, 1, 6),
    )
    ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    plt.close(fig)
    r = {
        "alpha": alpha,
        "steps": steps,
        "start": start,
        "bild": pic,
        "info": info,
        "table": table,
        "last": None,
        "lock": False,
        "graph_key": None,
        "fig": fig,
    }

    def readlinks():
        return (
            {j + 1: [i + 1 for i in range(4) if r["cells"][i][j].value] for j in range(4)}
            if editierbar
            else SKRIPTLINKS
        )

    def update(_=None):
        if r["lock"]:
            return
        links = readlinks()
        G = googlematrix(links, alpha.value / 100)
        q = stationaer_eindeutig(G)
        ps = entwicklung(G, np.eye(size)[:, start.value - 1], steps.value)
        pk = ps[-1]
        if links != r["graph_key"]:
            netzbild(graphax, links)
            r["graph_key"] = links
        for i, (bq, bp) in enumerate(zip(barsq, barsp)):
            bq.set_visible(q is not None)
            bq.set_height(0 if q is None else q[i])
            bp.set_height(pk[i])
        ax.legend(
            [barsp] if q is None else [barsq, barsp],
            (
                ["Verteilung nach k Schritten"]
                if q is None
                else ["Stationäres q", "Verteilung nach k Schritten"]
            ),
            fontsize=8,
            loc="upper right",
        )
        zeichnen(fig, pic)
        info.value = ranginfo(q)
        if q is not None:
            info.value += f" Größter Abstand von p nach {steps.value} Schritten zu q: <b>{np.max(abs(pk-q)):.4f}</b>."
        if alpha.value == 100:
            info.value += " Bei 100 % ist Konvergenz aus jedem Start nicht garantiert."
        table.value = matrixtext(G) + (
            tabelle(
                ["Seite", "Stationäres Gewicht"], [[i + 1, f"{v:.6f}"] for i, v in enumerate(q)]
            )
            if q is not None
            else ""
        )
        r["last"] = {"links": links, "matrix": G, "q": q, "distributions": ps}

    leading = []
    if editierbar:
        cells = [
            [
                W.Checkbox(
                    value=(i + 1 in MINILINKS[j + 1]),
                    indent=False,
                    layout=W.Layout(width="50px"),
                )
                for j in range(4)
            ]
            for i in range(4)
        ]
        r["cells"] = cells
        widgets = [W.HTML("Ziel ↓<br>Start →")] + [W.HTML(f"<b>{j}</b>") for j in range(1, 5)]
        for i, row in enumerate(cells):
            widgets += [W.HTML(f"<b>{i+1}</b>")] + row
        grid = W.GridBox(
            widgets, layout=W.Layout(grid_template_columns="repeat(5,65px)", grid_gap="4px")
        )
        reset = W.Button(
            description="Startnetz wiederherstellen", layout=W.Layout(width="240px")
        )

        def resetnet(_):
            r["lock"] = True
            try:
                for i, row in enumerate(cells):
                    for j, w in enumerate(row):
                        w.value = i + 1 in MINILINKS[j + 1]
            finally:
                r["lock"] = False
            update()

        reset.on_click(resetnet)
        r["reset"] = reset
        for row in cells:
            for w in row:
                w.observe(update, names="value")
        leading = [W.HTML("Häkchen = Link von der Spalte zur Zeile"), grid, reset]
    for w in (alpha, steps, start):
        w.observe(update, names="value")
    display(
        W.VBox(
            leading
            + [
                feld("Links folgen: α in %", alpha),
                feld("Schritte k für den Vergleich mit q", steps),
                start,
                pic,
                info,
                extra,
            ]
        )
    )
    update()
    r["update"] = update
    PANELS["netz" if editierbar else "skript"] = r
    return r


def station_pagerank_mc():
    n = W.SelectionSlider(
        options=[30, 100, 300, 1000, 3000, 10000, 30000, 100000],
        value=30,
        continuous_update=True,
        layout=W.Layout(width="480px", max_width="100%"),
    )
    start = W.Dropdown(options=list(range(1, 11)), value=1, description="Startseite:")
    btn = W.Button(
        description="Neue Surf-Tour auslosen",
        button_style="info",
        layout=W.Layout(width="240px"),
    )
    show = W.Checkbox(
        value=False,
        description="PageRank q zum Vergleichen einblenden",
        indent=False,
        layout=W.Layout(width="390px"),
    )
    pic = bild()
    info = W.HTML()
    G = googlematrix(SKRIPTLINKS, 0.85)
    q = stationaer_eindeutig(G)
    fig, ax = plt.subplots(figsize=(6.7, 4.6))
    plt.close(fig)
    r = {
        "n": n,
        "start": start,
        "show": show,
        "button": btn,
        "bild": pic,
        "seed": 2026,
        "key": None,
        "last": None,
        "fig": fig,
    }

    def update(_=None):
        key = (start.value, r["seed"])
        if r["key"] != key:
            r["path"] = pfad_ziehen(G, start.value - 1, 100000, r["seed"])
            r["counts_cum"] = np.array(
                [np.cumsum(r["path"][1:] == i, dtype=np.int32) for i in range(10)]
            )
            r["key"] = key
        N = n.value
        counts = r["counts_cum"][:, N - 1]
        freq = counts / N
        # A tie receives the same rank; page number only determines display order.
        order = np.lexsort((np.arange(10), -counts))
        ranks = 1 + (counts[None, :] > counts[:, None]).sum(axis=1)
        ax.clear()
        y = np.arange(10)
        if show.value:
            ax.barh(y - 0.18, q[order], height=0.34, label="PageRank q", color="#2875aa")
            ax.barh(
                y + 0.18, freq[order], height=0.34, label="Beobachteter Anteil", color="#ad6500"
            )
        else:
            ax.barh(y, freq[order], height=0.6, label="Beobachteter Anteil", color="#ad6500")
        maximum = max(float(freq.max()), float(q.max()) if show.value else 0, 0.05)
        for row, i in enumerate(order):
            ax.text(
                max(freq[i], q[i] if show.value else 0) + 0.015 * maximum,
                row,
                f"{counts[i]} Bes.",
                va="center",
                fontsize=8,
            )
        ax.set(
            yticks=y,
            yticklabels=[f"Seite {i+1} · Rang {ranks[i]}" for i in order],
            xlim=(0, maximum * 1.36),
            xlabel="Besuchsanteil / Gewicht",
            title=f"Rangfolge nach {N:,} Besuchen".replace(",", " "),
        )
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.2)
        ax.set_axisbelow(True)
        ax.legend(loc="lower right", fontsize=8)
        fig.tight_layout()
        zeichnen(fig, pic)
        leaders = np.flatnonzero(counts == counts.max()) + 1
        info.value = f'<b>Vorn nach diesen Daten: Seite(n) {", ".join(map(str,leaders))}.</b> Gezählt: Besuche 1 bis {N}, ohne Startbesuch. Gleiche Besuchszahl bedeutet gleichen Rang; innerhalb eines Gleichstands stehen die Seitennummern aufsteigend.'
        if show.value:
            info.value += " Die Sortierung bleibt nach den beobachteten Anteilen; blau zeigt die stationären Gewichte zum Vergleich."
        info.value += "<br><b>Geschätzter Verteilungsvektor in der Reihenfolge 1 bis 10:</b>"
        info.value += tabelle(
            ["Seite", "Besuche", "Anteil"],
            [[i + 1, int(counts[i]), f"{freq[i]:.5f}"] for i in range(10)],
        )
        r["last"] = {
            "path": r["path"][: N + 1].copy(),
            "counts": counts.copy(),
            "frequencies": freq,
            "q": q.copy(),
            "order": order,
            "ranks": ranks,
            "N": N,
        }

    def new(_):
        r["seed"] += 1
        update()

    for w in (n, start, show):
        w.observe(update, names="value")
    btn.on_click(new)
    display(W.VBox([start, feld("Besuche eines einzelnen Surfers N", n), btn, show, pic, info]))
    update()
    PANELS["mc"] = r
    return r


LONDON_ANCHORS = {
    "Paddington": (308, 283),
    "Paddington (H&C)": (308, 268),
    "Edgware Road (Bakerloo)": (369, 226),
    "Edgware Road (Circle/District)": (354, 218),
    "Marylebone": (410, 191),
    "Baker Street": (459, 189),
    "Regent's Park": (540, 183),
    "Oxford Circus": (587, 289),
    "Piccadilly Circus": (646, 356),
    "Charing Cross": (723, 380),
    "Embankment": (743, 389),
    "Waterloo": (808, 437),
    "Lambeth North": (830, 497),
    "Elephant & Castle": (924, 559),
    "Notting Hill Gate": (136, 362),
    "Queensway": (212, 351),
    "Lancaster Gate": (308, 332),
    "Marble Arch": (447, 306),
    "Bond Street": (522, 299),
    "Tottenham Court Road": (672, 269),
    "Holborn": (762, 257),
    "Chancery Lane": (827, 242),
    "St. Paul's": (949, 290),
    "Bank": (1018, 311),
    "Monument": (1037, 343),
    "Liverpool Street": (1069, 252),
    "Tower Hill": (1116, 356),
    "Aldgate": (1125, 298),
    "Aldgate East": (1148, 284),
    "Moorgate": (1018, 242),
    "Barbican": (942, 218),
    "Farringdon": (882, 219),
    "King's Cross St. Pancras": (730, 81),
    "Euston Square": (631, 145),
    "Great Portland Street": (566, 173),
    "Bayswater": (208, 327),
    "High Street Kensington": (171, 473),
    "Gloucester Road": (249, 556),
    "South Kensington": (323, 562),
    "Sloane Square": (464, 584),
    "Victoria": (560, 530),
    "St. James's Park": (651, 492),
    "Westminster": (718, 471),
    "Temple": (810, 340),
    "Blackfriars": (900, 327),
    "Mansion House": (974, 326),
    "Cannon Street": (1003, 337),
    "Earl's Court": (131, 589),
    "Green Park": (576, 397),
    "Southwark": (883, 433),
    "London Bridge": (1036, 417),
    "Euston": (651, 117),
    "Warren Street": (611, 164),
    "Goodge Street": (645, 214),
    "Leicester Square": (695, 337),
    "Angel": (876, 65),
    "Old Street": (1028, 142),
    "Borough": (973, 471),
    "Knightsbridge": (428, 463),
    "Hyde Park Corner": (497, 453),
    "Covent Garden": (725, 313),
    "Russell Square": (725, 183),
    "Vauxhall": (718, 666),
    "Pimlico": (654, 620),
}


LONDON_ROUTES = {
    "Bakerloo": "Paddington|Edgware Road (Bakerloo)|Marylebone|Baker Street|Regent's Park|Oxford Circus|Piccadilly Circus|Charing Cross|Embankment|Waterloo|Lambeth North|Elephant & Castle",
    "Central": "Notting Hill Gate|Queensway|Lancaster Gate|Marble Arch|Bond Street|Oxford Circus|Tottenham Court Road|Holborn|Chancery Lane|St. Paul's|Bank|Liverpool Street",
    "Circle": "Tower Hill|Aldgate|Liverpool Street|Moorgate|Barbican|Farringdon|King's Cross St. Pancras|Euston Square|Great Portland Street|Baker Street|Edgware Road (Circle/District)|Paddington|Bayswater|Notting Hill Gate|High Street Kensington|Gloucester Road|South Kensington|Sloane Square|Victoria|St. James's Park|Westminster|Embankment|Temple|Blackfriars|Mansion House|Cannon Street|Monument|Tower Hill",
    "District, Earl’s Court–Tower Hill": "Earl's Court|Gloucester Road|South Kensington|Sloane Square|Victoria|St. James's Park|Westminster|Embankment|Temple|Blackfriars|Mansion House|Cannon Street|Monument|Tower Hill|Aldgate East",
    "District, Kensington": "Earl's Court|High Street Kensington|Notting Hill Gate|Bayswater|Paddington|Edgware Road (Circle/District)",
    "Hammersmith & City": "Paddington (H&C)|Edgware Road (Circle/District)|Baker Street|Great Portland Street|Euston Square|King's Cross St. Pancras|Farringdon|Barbican|Moorgate|Liverpool Street|Aldgate East",
    "Metropolitan": "Baker Street|Great Portland Street|Euston Square|King's Cross St. Pancras|Farringdon|Barbican|Moorgate|Liverpool Street|Aldgate",
    "Jubilee": "Baker Street|Bond Street|Green Park|Westminster|Waterloo|Southwark|London Bridge",
    "Northern, Charing Cross": "Euston|Warren Street|Goodge Street|Tottenham Court Road|Leicester Square|Charing Cross|Embankment|Waterloo",
    "Northern, Bank": "King's Cross St. Pancras|Angel|Old Street|Moorgate|Bank|London Bridge|Borough|Elephant & Castle",
    "Piccadilly": "Earl's Court|Gloucester Road|South Kensington|Knightsbridge|Hyde Park Corner|Green Park|Piccadilly Circus|Leicester Square|Covent Garden|Holborn|Russell Square|King's Cross St. Pancras",
    "Victoria": "Vauxhall|Pimlico|Victoria|Green Park|Oxford Circus|Warren Street|Euston|King's Cross St. Pancras",
    "Waterloo & City": "Waterloo|Bank",
}


LONDON_ALIAS = {
    "Bank": "Bank / Monument",
    "Monument": "Bank / Monument",
    "Paddington (H&C)": "Paddington",
}


# Londoner Netz und Stationen.
def london_netz():
    names = sorted({LONDON_ALIAS.get(s, s) for s in LONDON_ANCHORS})
    index = {s: i for i, s in enumerate(names)}
    adj = np.zeros((len(names), len(names)), dtype=int)
    edges = {}
    for route, sequence in LONDON_ROUTES.items():
        stops = sequence.split("|")
        for a, b in zip(stops, stops[1:]):
            i, j = index[LONDON_ALIAS.get(a, a)], index[LONDON_ALIAS.get(b, b)]
            if i == j:
                continue
            adj[i, j] = adj[j, i] = 1
            key = tuple(sorted((i, j)))
            edges.setdefault(
                key, {"routes": [], "anchors": {i: LONDON_ANCHORS[a], j: LONDON_ANCHORS[b]}}
            )
            edges[key]["routes"].append(route)
    anchors = []
    owners = []
    for name, xy in LONDON_ANCHORS.items():
        anchors.append(xy)
        owners.append(index[LONDON_ALIAS.get(name, name)])
    degree = adj.sum(axis=0)
    M = adj / degree[None, :]
    q = degree / degree.sum()
    return {
        "names": names,
        "index": index,
        "adj": adj,
        "degree": degree,
        "M": M,
        "q": q,
        "edges": edges,
        "xy": np.array(anchors),
        "owners": np.array(owners),
    }


LONDON = london_netz()


def london_pfad(start, n=100000, seed=2026):
    rng = np.random.default_rng(seed)
    u = rng.random(n)
    neighbors = [np.flatnonzero(row) for row in LONDON["adj"]]
    x = np.empty(n + 1, dtype=np.int16)
    x[0] = LONDON["index"][start]
    for t, v in enumerate(u, 1):
        choices = neighbors[x[t - 1]]
        x[t] = choices[min(int(v * len(choices)), len(choices) - 1)]
    return x


def london_counts(path, n):
    return np.bincount(path[1 : n + 1], minlength=len(LONDON["names"]))


def london_karte():
    import matplotlib.patheffects as pe

    fig = plt.figure(figsize=(12.8, 8), dpi=100)
    gray = np.dot(LONDON_PNG[..., :3], [0.2126, 0.7152, 0.0722])
    ax = fig.add_axes([0, 0.09, 1, 0.91])
    ax.imshow(0.2 + 0.8 * gray, cmap="gray", vmin=0, vmax=1, extent=(0, 1280, 730, 0))
    ax.set(xlim=(0, 1280), ylim=(730, 0))
    ax.axis("off")
    xy = LONDON["xy"]
    ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=175,
        facecolors="white",
        edgecolors="white",
        linewidths=2,
        zorder=3,
    )
    points = ax.scatter(
        xy[:, 0],
        xy[:, 1],
        s=95,
        c=np.zeros(len(xy)),
        cmap="YlOrRd",
        vmin=0,
        vmax=0.08,
        edgecolors="#171717",
        linewidths=1,
        zorder=4,
    )
    selected = ax.scatter(
        [],
        [],
        s=400,
        marker="D",
        facecolors="none",
        edgecolors="#681491",
        linewidths=3,
        zorder=8,
    )
    neighbors = ax.scatter(
        [],
        [],
        s=280,
        marker="s",
        facecolors="none",
        edgecolors="#006c42",
        linewidths=2.8,
        zorder=7,
    )
    for artist in (selected, neighbors):
        artist.set_path_effects([pe.Stroke(linewidth=6.5, foreground="white"), pe.Normal()])
    recent = ax.plot([], [], color="#681491", ls="--", lw=2.5, zorder=5)[0]
    recent.set_path_effects([pe.Stroke(linewidth=5.5, foreground="white"), pe.Normal()])
    cax = fig.add_axes([0.31, 0.06, 0.38, 0.018])
    bar = fig.colorbar(points, cax=cax, orientation="horizontal", extend="max")
    pic = W.Image(format="png", layout=W.Layout(width="1280px", max_width="100%"))
    plt.close(fig)
    return {
        "fig": fig,
        "ax": ax,
        "points": points,
        "selected": selected,
        "neighbors": neighbors,
        "recent": recent,
        "bar": bar,
        "image": pic,
    }


def london_markers(indices):
    mask = np.isin(LONDON["owners"], list(indices))
    return LONDON["xy"][mask]


def london_render(view, values, mode="Anteil", selected=(), neighbors=(), edge=None):
    view["points"].set_array(np.asarray(values)[LONDON["owners"]])
    if mode == "Grad":
        view["points"].set_clim(0, 7)
        view["bar"].set_ticks([0, 1, 2, 3, 4, 5, 6, 7])
        view["bar"].set_ticklabels([str(i) for i in range(8)])
        view["bar"].set_label("Anzahl verschiedener Nachbarstationen", fontsize=10)
    else:
        view["points"].set_clim(0, 0.08)
        view["bar"].set_ticks([0, 0.02, 0.04, 0.06, 0.08])
        view["bar"].set_ticklabels(["0 %", "2 %", "4 %", "6 %", "≥ 8 %"])
        view["bar"].set_label(
            "Besuchsanteil" if mode == "Anteil" else "Stationärer Anteil q", fontsize=10
        )
    view["selected"].set_offsets(london_markers(selected))
    view["neighbors"].set_offsets(london_markers(neighbors))
    if edge is None:
        view["recent"].set_data([], [])
    else:
        i, j = edge
        pos = LONDON["edges"][tuple(sorted((int(i), int(j))))]["anchors"]
        a, b = pos[int(i)], pos[int(j)]
        view["recent"].set_data([a[0], b[0]], [a[1], b[1]])
    with io.BytesIO() as buffer:
        view["fig"].savefig(buffer, format="png", dpi=100)
        view["image"].value = buffer.getvalue()


def london_auswahl(value):
    return W.Dropdown(
        options=LONDON["names"], value=value, layout=W.Layout(width="370px", max_width="100%")
    )


def station_london_fahrt():
    start = london_auswahl("Vauxhall")
    horizon = W.Dropdown(
        options=[100, 1000, 10000, 100000], value=1000, description="Bis Schritt:"
    )
    n = W.IntSlider(
        value=0,
        min=0,
        max=1000,
        continuous_update=True,
        layout=W.Layout(width="600px", max_width="100%"),
    )
    play = W.Play(value=0, min=0, max=1000, step=1, interval=450, repeat=False)
    link = W.jslink((play, "value"), (n, "value"))
    btn = W.Button(
        description="Neue Zufallsfahrt", button_style="info", layout=W.Layout(width="190px")
    )
    info = W.HTML()
    ranking = W.HTML()
    local = W.HTML()
    returns = W.HTML()
    view = london_karte()
    r = {
        "start": start,
        "horizon": horizon,
        "n": n,
        "play": play,
        "link": link,
        "button": btn,
        "view": view,
        "seed": 2026,
        "key": None,
        "lock": False,
        "last": None,
    }

    def update(_=None):
        if r["lock"]:
            return
        key = (start.value, r["seed"])
        if r["key"] != key:
            r["path"] = london_pfad(start.value, seed=r["seed"])
            r["key"] = key
        N = n.value
        x = r["path"]
        counts = london_counts(x, N)
        freq = counts / N if N else np.zeros(len(counts))
        current = int(x[N])
        s = LONDON["index"][start.value]
        neighbors = np.flatnonzero(LONDON["adj"][s])
        degree = LONDON["degree"]
        local.value = tabelle(
            ["Nachbarn der Startstation", "Anzahl ihrer Nachbarn"],
            [[LONDON["names"][j], int(degree[j])] for j in neighbors],
        )
        edge = (x[N - 1], x[N]) if N else None
        london_render(view, freq, selected=[current], edge=edge)
        line = (
            ("Start: " + LONDON["names"][current])
            if not N
            else (LONDON["names"][int(x[N - 1])] + " → " + LONDON["names"][current])
        )
        info.value = f"<b>Schritt {N}: {html.escape(line)}</b><br>Violette Raute: aktuelle Station. Gestrichelt: letzter Übergang (schematisch). Farben: Besuchsanteile; ab 8 % gesättigt."
        eligible = x[: N - 1] == s if N >= 2 else np.zeros(0, dtype=bool)
        num = int(eligible.sum())
        back = int(np.sum(eligible & (x[2 : N + 1] == s))) if N >= 2 else 0
        returns.value = (
            f"<b>Rückkehr nach genau zwei Schritten zur Startstation:</b> {back} / {num} = {back/num:.2%}."
            if num
            else "Noch kein auswertbarer Zweischritt-Versuch ab der Startstation."
        )
        returns.value += "<br>Ausgewertet werden alle Zeiten t mit Xₜ = Startstation und t + 2 ≤ n; die Versuche können sich überlappen und sind nicht unabhängig."
        if not N:
            ranking.value = "Noch kein Besuch gezählt; Schritt 0 zählt nicht mit."
        else:
            leaders = sorted(
                range(len(counts)), key=lambda i: (-counts[i], LONDON["names"][i])
            )[:8]
            ranking.value = tabelle(
                ["Häufig besucht (max. 8)", "Besuche", "Anteil"],
                [
                    [LONDON["names"][i], int(counts[i]), f"{freq[i]:.2%}"]
                    for i in leaders
                    if counts[i] > 0
                ],
            )
            ranking.value += f"<br>Besuchsanteil der Startstation {html.escape(start.value)}: <b>{freq[s]:.3%}</b>."
        r["last"] = {
            "counts": counts,
            "freq": freq,
            "current": current,
            "n": N,
            "eligible": num,
            "returns": back,
            "start_index": s,
        }

    def limit(_):
        r["lock"] = True
        play.max = horizon.value
        n.max = horizon.value
        n.value = horizon.value
        play.value = n.value
        r["lock"] = False
        update()

    def new(_):
        r["seed"] += 1
        update()

    horizon.observe(limit, names="value")
    start.observe(update, names="value")
    n.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("Startstation", start),
                local,
                W.HBox([horizon, btn]),
                feld("Gleiche Fahrt verlängern oder zurückspulen", n),
                play,
                returns,
                info,
                view["image"],
                ranking,
            ]
        )
    )
    update()
    PANELS["london_fahrt"] = r
    return r


def station_london_verteilung():
    start = london_auswahl("Bank / Monument")
    selected = london_auswahl("Green Park")
    n = W.IntSlider(
        value=10000,
        min=100,
        max=100000,
        step=100,
        continuous_update=True,
        layout=W.Layout(width="600px", max_width="100%"),
    )
    mode = W.Dropdown(
        options=["Besuchsanteile", "Nachbarzahlen", "Stationäres q (zum Prüfen)"],
        value="Nachbarzahlen",
        layout=W.Layout(width="300px"),
    )
    btn = W.Button(
        description="Neue Zufallsfahrt", button_style="info", layout=W.Layout(width="190px")
    )
    view = london_karte()
    info = W.HTML()
    proof = W.HTML()
    detail = W.Image(format="png", layout=W.Layout(width="920px", max_width="100%"))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
    plt.close(fig)
    r = {
        "start": start,
        "selected": selected,
        "n": n,
        "mode": mode,
        "button": btn,
        "view": view,
        "detail": detail,
        "info": info,
        "seed": 2026,
        "key": None,
        "last": None,
        "fig": fig,
    }

    def update(_=None):
        key = (start.value, r["seed"])
        if r["key"] != key:
            r["path"] = london_pfad(start.value, seed=r["seed"])
            r["key"] = key
        N = n.value
        counts = london_counts(r["path"], N)
        freq = counts / N
        i = LONDON["index"][selected.value]
        neighbors = np.flatnonzero(LONDON["adj"][:, i])
        q = LONDON["q"]
        degree = LONDON["degree"]
        theoretical = mode.value.startswith("Stationäres")
        values = q if theoretical else (degree if mode.value == "Nachbarzahlen" else freq)
        london_render(
            view,
            values,
            mode=(
                "Stationär"
                if theoretical
                else ("Grad" if mode.value == "Nachbarzahlen" else "Anteil")
            ),
            selected=[i],
            neighbors=neighbors,
        )
        info.value = (
            f"<b>{html.escape(selected.value)}</b>: {degree[i]} verschiedene Nachbarn (grüne Quadrate): "
            + html.escape(", ".join(LONDON["names"][j] for j in neighbors))
            + f".<br>Beobachtet: {counts[i]} von {N} Besuchen ({freq[i]:.3%}). Summe aller Nachbarzahlen: {degree.sum()}."
        )
        if theoretical:
            info.value += f"<br>Stationärer Anteil dieser Station: <b>{q[i]:.3%}</b>."
        for ax in axes:
            ax.clear()
            ax.grid(alpha=0.2)
        # Exact integer x coordinates. Coincident observations are combined, not jittered.
        groups = {}
        for j in range(len(degree)):
            groups.setdefault((int(degree[j]), int(counts[j])), []).append(j)
        keys = list(groups)
        ids = [groups[key][0] for key in keys]
        sizes = [len(groups[key]) for key in keys]
        scatter = axes[0].scatter(
            [key[0] for key in keys],
            [key[1] / N for key in keys],
            s=[40 + 22 * (k - 1) for k in sizes],
            c=values[ids],
            cmap=view["points"].cmap,
            norm=view["points"].norm,
            edgecolors="#333333",
            linewidths=0.7,
            zorder=3,
        )
        axes[0].scatter(
            degree[i],
            freq[i],
            s=140,
            marker="D",
            facecolors="none",
            edgecolors="#681491",
            linewidth=2,
            zorder=5,
        )
        if theoretical:
            ds = np.arange(1, 8)
            axes[0].plot(ds, ds / degree.sum(), color="#216da6", label="Stationäre Vorhersage")
            axes[0].legend(fontsize=8)
        axes[0].set(
            xlim=(0.65, 7.35),
            xticks=range(1, 8),
            xlabel="Anzahl verschiedener Nachbarn",
            ylabel="Besuchsanteil",
            title="Stationen nach Nachbarzahl",
        )
        ends = np.unique(np.geomspace(1, N, min(N, 500)).astype(int))
        cum = np.cumsum(r["path"][1 : N + 1] == i)
        running = cum[ends - 1] / ends
        axes[1].plot(ends, running, color="#c36916", label="Beobachtet")
        if theoretical:
            axes[1].axhline(q[i], color="#216da6", ls="--", label="Stationär")
        axes[1].set(
            xscale="log",
            xlim=(1, N),
            xlabel="Besuche (logarithmisch)",
            ylabel="Laufender Besuchsanteil",
            title="Ausgewählte Station",
        )
        axes[1].legend(fontsize=8)
        same = groups[(int(degree[i]), int(counts[i]))]
        if len(same) > 1:
            info.value += (
                "<br>Gleicher Punkt im linken Diagramm: "
                + html.escape(", ".join(LONDON["names"][j] for j in same))
                + "."
            )
        fig.tight_layout()
        with io.BytesIO() as buffer:
            fig.savefig(buffer, format="png", dpi=100)
            detail.value = buffer.getvalue()
        proof.value = "Farben wie auf der Karte und ihrer Skala. Größere Punkte bündeln Stationen mit exakt gleicher Nachbarzahl und Besuchszahl."
        r["last"] = {
            "counts": counts,
            "freq": freq,
            "q": q.copy(),
            "degree": degree.copy(),
            "selected": i,
            "neighbors": neighbors,
            "running": running,
            "ends": ends,
            "theoretical": theoretical,
            "scatter": scatter,
            "groups": groups,
        }

    def new(_):
        r["seed"] += 1
        update()

    for w in (start, selected, n, mode):
        w.observe(update, names="value")
    btn.on_click(new)
    display(
        W.VBox(
            [
                feld("Startstation des eigenen Versuchs", start),
                feld("Untersuchte Station (violette Raute)", selected),
                feld("Anzahl ausgewerteter Besuche", n),
                W.HBox([mode, btn]),
                info,
                view["image"],
                detail,
                proof,
            ]
        )
    )
    update()
    PANELS["london_verteilung"] = r
    return r


# Eingebetteter Netzplan: Die Bilddaten benötigen keine zusätzliche Datei.

LONDON_PNG_BASE64 = (
    "..."
)

LONDON_PNG = plt.imread(io.BytesIO(base64.b64decode(LONDON_PNG_BASE64)), format="png")

print("Bereit – weiter mit Station 1.")


<br><br>

## 1 · London: Wie wahrscheinlich ist eine schnelle Rückkehr?

An jedem Halt wählen wir **gleichverteilt eine verschiedene Nachbarstation**; die Rückfahrt ist erlaubt. Die kleine Tabelle nennt die Nachbarn der Startstation und deren Nachbarzahlen. Die Karte färbt die einzelnen Stationen in Abhängigkeit davon ein, wie häufig sie besucht wurden. Die **violette Raute** zeigt den aktuellen Halt.

1. **Rechne vor der Fahrt:** Starte gedanklich in **Vauxhall**, dann in **Green Park**. Wie groß ist jeweils die Wahrscheinlichkeit, nach **genau zwei Schritten** wieder am Start zu sein? Zähle alle möglichen Hin-und-zurück-Wege und addiere ihre Wahrscheinlichkeiten. Die Nachbartabelle hilft dir dabei. <span style="color: #707070;">(leicht)</span>
2. Wähle die Starts "Vauxhall" und "Green Park" im Werkzeug und verlängere die Fahrt auf **10 000 Schritte**. Vergleiche deine Ergebnisse mit den beobachteten Zweischritt-Rückkehrquoten. Gezählt wird jede auswertbare Abfahrt von der Startstation (Vauxhall oder Green Park) innerhalb dieser Fahrt. Prüfe auch eine neu ausgeloste Fahrt. <span style="color: #707070;">(leicht)</span>
3. Bedeutet eine größere Zweischritt-Rückkehrwahrscheinlichkeit zugleich einen größeren **langfristigen Besuchsanteil**? Vergleiche die beiden Startstationen bei langen Fahrten. Die nächste Station untersucht, wovon dieser Anteil abhängt. <span style="color: #707070;">(schwerer)</span>

Der Schrittregler spult dieselbe Fahrt vor und zurück; Start/Pause spielt kurze Abschnitte ab. Die gestrichelte Verbindung zeigt den letzten Übergang schematisch.


In [ ]:
station_1 = station_london_fahrt()

<br><br>

## 2 · Warum werden manche Stationen häufiger besucht?

Die violette Raute markiert die untersuchte Station, grüne Quadrate ihre Nachbarn.

1. Vergleiche bei einer langen Fahrt die Besuchsanteile von **Green Park, Vauxhall und zwei Stationen mit derselben Nachbarzahl**. Welchen Zusammenhang mit der Nachbarzahl erkennst du? Wähle anschließend **„Stationäres q (zum Prüfen)“**. Müssen Stationen mit gleichem $q_i$ schon in einer endlichen Fahrt gleich viele Besuche haben? <span style="color: #707070;">(leicht)</span>
2. Vermute eine Formel für $q_i$ aus der Nachbarzahl $d_i$ und normiere sie auf Summe 1. Prüfe deine Formel an den angezeigten stationären Gewichten. <span style="color: #707070;">(schwerer)</span>
3. **Begründe deine Formel auf Papier:** Für benachbarte Stationen i und j gilt $m_{ij}=1/d_j$ und $m_{ji}=1/d_i$. Setze deinen Kandidaten für q in beide Seiten von $q_jm_{ij}=q_im_{ji}$ ein und vergleiche. Prüfe die Gleichung auch für nicht benachbarte Stationen. Diese Gleichheit heißt **detaillierte Balance**: Im stationären Zustand sind die Wahrscheinlichkeitsströme in beide Richtungen gleich. <span style="color: #707070;">(schwerer)</span>
4. **Zeige, warum aus der detaillierten Balance folgt, dass q stationär ist:** Summiere die Gleichung bei festem i über alle j. Nutze die Spaltensumme $\sum_jm_{ji}=1$, um $(Mq)_i=q_i$ herzuleiten. Warum konvergieren hier außerdem die Zeitanteile gegen q? Begründe deine Antwort mithilfe der Irreduzibilität. <span style="color: #707070;">(schwer)</span>


In [ ]:
station_2 = station_london_verteilung()

<br><br>

## 3 · Vom Link zum PageRank: Ein Mini-Internet

Wir betrachten nun ein Mini-Internet mit vier Webseiten und folgenden Links:

| Seite | Linkziele |
|---|---|
| 1 | 2, 3 |
| 2 | 3 |
| 3 | 1 |
| 4 | keine |

Mit Wahrscheinlichkeit $\alpha$ folgen wir einem gleichverteilt zufällig gewählten Link; mit Wahrscheinlichkeit $1 - \alpha$ wählen wir irgendeine der vier Seiten zufällig gleichverteilt. Falls eine Seite gar keine Links beinhaltet, wird auch im ersten Fall gleichverteilt eine zufällige Seite gewählt. Mit n Seiten lautet die Übergangsmatrix eines solchen Internetsurfers also $M=\alpha H+(1-\alpha)J/n$, wobei H die Wahrscheinlichkeit gleichmäßig auf die Linkziele verteilt (bei fehlenden Links auf alle Seiten), und J eine Matrix bestehend aus Einsen ist. Eine stationäre Verteilung q von M bildet dann den **PageRank** der Webseiten.

1. Stelle die Matrix M für das Mini-Internet mit $\alpha = 0{,}85$ auf. <span style="color: #707070;">(schwerer)</span>
2. Verändere nun einzelne Links und des Mini-Internets sowie den Parameter $\alpha$. Ein Häkchen setzt einen Link **von der Spalte zur Zeile**. Die Grafik zeigt die resultierende stationäre Verteilung (also die PageRank-Gewichte) in blau, sowie die Verteilung nach k Schritten ab der gewählten Startseite in orange. Welche einzelne Linkänderung erhöht das Gewicht von Seite 4? Vermute zuerst und prüfe dann. <span style="color: #707070;">(leicht)</span>
3. Vergleiche **0 %, 85 % und 100 %** für $\alpha$. Baue bei 100 % erst **nur Selbstlinks**, dann den Zyklus **1 → 2 → 3 → 4 → 1**. Gibt es jeweils ein eindeutiges q? Konvergiert die Verteilung aus jedem Start gegen q? <span style="color: #707070;">(schwerer)</span>
4. Für $0\leq\alpha<1$ sind alle Einträge von M positiv. Begründe daraus Irreduzibilität und Aperiodizität. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_3 = pagerank_labor(editierbar=True)

<br><br>

## 4 · Das Zehn-Seiten-Netz aus dem Skript

Für **Beispiel 2.24** im Skript verwenden wir diese Links:

| Seite | Linkziele | Seite | Linkziele |
|---|---|---|---|
| 1 | 2 | 6 | 2, 5 |
| 2 | 3 | 7 | 2, 5 |
| 3 | 2 | 8 | 2, 5 |
| 4 | 1, 2 | 9 | 5 |
| 5 | 2, 4, 6 | 10 | 5 |

1. **Welche Seite erwartest du bei α = 85 % an erster Stelle?** Vergleiche besonders Seiten 3 und 5: Genügt die Anzahl eingehender Links als Erklärung? <span style="color: #707070;">(leicht)</span>
2. Variiere α, Startseite und Schrittzahl. Die Balken haben dieselbe Bedeutung wie in Station 3. Die Matrix und Gewichte lassen sich aufklappen. <span style="color: #707070;">(leicht)</span>
3. Untersuche bei **α = 100 %** den Verlauf ab Seite 1 für gerade und ungerade Schrittzahlen. Bedeutet ein eindeutiges q automatisch Konvergenz? <span style="color: #707070;">(schwerer)</span>
4. Für α = 85 % steht in Zeile 4 und Spalte 5 der Übergangsmatrix M der Eintrag 0.2983. Rechne diesen Eintrag auf dem Papier nach. <span style="color: #707070;">(schwerer)</span>


In [ ]:
station_4 = pagerank_labor(editierbar=False)

<br><br>

## 5 · Welche Seite liegt wirklich vorn? (Skriptaufgabe 2.25.2)

Ein einzelner Surfer bewegt sich durch das Zehn-Seiten-Netz aus Station 4; **α = 0,85** bleibt fest. Wir zählen seine Besuche und ordnen die Seiten nach ihrer Häufigkeit. Der Regler verlängert **dieselbe Tour**. Gleiche Besuchszahlen erhalten denselben Rang; der Startbesuch zählt nicht mit.

1. **Beginne mit N = 30.** Welche drei Seiten stehen vorn? Verlängere auf **100, 1 000 und 10 000 Besuche**: Welche Plätze ändern sich noch? Notiere bei **10 000 Besuchen** den geschätzten Verteilungsvektor in der Reihenfolge der Seiten 1 bis 10 (Tabelle unter der Grafik). Prüfe deinen Vektor und die Rangfolge erst danach durch Einblenden von q. <span style="color: #707070;">(leicht)</span>
2. **Verlässlicher Spitzenreiter?** Wähle N = 100 und lose drei neue Touren aus. Gewinnt immer dieselbe Seite? Wiederhole bei N = 10 000. Warum hat Seite 5 keinen sonderlich hohen PageRank, obwohl viele Links auf sie verweisen? <span style="color: #707070;">(schwerer)</span>
3. **Gleichstand:** Betrachte die Seiten **7, 8, 9 und 10** im Netz aus Station 4. Führen überhaupt Links auf diese Seiten? Berechne auf dem Papier ihren PageRank und vergleiche die vier Werte. Müssen sie deshalb in einer endlichen Tour gleich oft besucht werden? Prüfe mit den Besuchszahlen und blende anschließend q ein. <span style="color: #707070;">(schwerer)</span>

Die Zahlen neben den Balken sind Besuchszahlen. Orange zeigt den beobachteten Anteil; Blau lässt sich als Vergleich mit dem berechneten PageRank zuschalten.


In [ ]:
station_5 = station_pagerank_mc()

<br><br>

## Kleine Codeaufgabe · Die Sprungregel selbst umsetzen (optional) <span style="color: #707070;">(schwerer)</span>

Ergänze `M_eigen` mit der Formel $M=\alpha H+(1-\alpha)J/n$. `H` gehört zum festen Vier-Seiten-Startnetz; `J` enthält nur Einsen. In Python stehen `*` und `/` für Multiplikation und Division mit einer Zahl.

Die Zelle prüft deine Umsetzung und zeigt die Matrix. Vergleiche die erste Spalte mit deiner Rechnung aus Aufgabe 3. Ändere anschließend `alpha` auf 0 und 1. Hier werden **Dezimalzahlen** verwendet: `0.85` bedeutet 85 %.

In [ ]:
alpha = 0.85
n = 4
H = linkmatrix(MINILINKS)
J = np.ones((n, n))
M_eigen = None  # Ergänze hier deine Formel.

if not 0 <= alpha <= 1:
    print("alpha muss zwischen 0 und 1 liegen.")
elif M_eigen is None:
    print("Ersetze None durch deine Formel für G.")
elif np.shape(M_eigen) != (n, n):
    print("G braucht vier Zeilen und vier Spalten.")
else:
    print("Deine Matrix:\n", M_eigen)
    print("Spaltensummen:", np.sum(M_eigen, axis=0))
    print("Formel richtig umgesetzt:", np.allclose(
        M_eigen, googlematrix(MINILINKS, alpha), atol=1e-12, rtol=0))

<details>
<summary>Kartenquelle</summary>
    
> [London Underground Zone 1, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:London_Underground_Zone_1.svg), London Underground geographic maps project, ed g2s und James D. Forrester. Darstellung geändert: grauer Hintergrund, farbige Datenpunkte und Markierungen. Diese Kartenansichten ebenfalls unter CC BY-SA 3.0. Originalkarte unter [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/). 
</details>